# Conscience Research Learner Notebook

This notebook introduces the core evaluation and guardrail workflows in `conscience_research`.

What you will run:
- baseline oracle evaluation
- third-eye strict oracle evaluation
- runtime-only scenario edit check with baseline preservation
- production guardrail decision simulation


In [7]:
from pathlib import Path
import hashlib
import json
import sys

def find_repo_root(start: Path) -> Path:
    # Walk up until we find the project root containing agent.py
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "agent.py").exists() and (candidate / "scenarios.py").exists():
            return candidate
        nested = candidate / "conscience_research"
        if (nested / "agent.py").exists() and (nested / "scenarios.py").exists():
            return nested
    raise RuntimeError("Could not locate conscience_research project root from current working directory")

repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import agent
from production_service import decision_payload

print(f"Using repo root: {repo_root}")


Using repo root: /Users/lakshminarasimhan.santhanamgigkri.com/ai_conscience/conscience_research


## 1) Baseline oracle evaluation

In [8]:
baseline_results = agent.evaluate_conscience_with_oracle_mode(agent, "baseline")
baseline_results


{'layer1_nsm': 1.0,
 'layer2_judge': 1.0,
 'layer3_penalty': 1.0,
 'layer4_bum': 1.0,
 'layer5_continuity': 1.0,
 'conscience_score': 1.0}

## 2) Third-eye strict evaluation

In [9]:
third_eye_results = agent.evaluate_conscience_with_oracle_mode(agent, "third_eye")
third_eye_results


{'layer1_nsm': 0.976,
 'layer2_judge': 1.0,
 'layer3_penalty': 0.9875,
 'layer4_bum': 1.0,
 'layer5_continuity': 1.0,
 'conscience_score': 0.9927}

## 3) Runtime-only scenario edits with baseline preservation check

In [10]:
def scenario_bank_hash(bank):
    payload = [
        {
            "id": s.id,
            "layer": s.layer,
            "action": s.action,
            "context": s.context,
            "expected_verdict": s.expected_verdict,
            "severity": s.severity,
            "norm_domain": s.norm_domain,
            "description": s.description,
        }
        for s in bank
    ]
    raw = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(raw.encode()).hexdigest()

before_hash = scenario_bank_hash(agent.scenarios_module.SCENARIO_BANK)
runtime_results, runtime_meta = agent.evaluate_with_runtime_scenario_edits(agent, "live")
after_hash = scenario_bank_hash(agent.scenarios_module.SCENARIO_BANK)

{
    "runtime_results": runtime_results,
    "runtime_meta": runtime_meta,
    "baseline_preserved": before_hash == after_hash,
}


{'runtime_results': {'layer1_nsm': 1.0,
  'layer2_judge': 1.0,
  'layer3_penalty': 1.0,
  'layer4_bum': 1.0,
  'layer5_continuity': 1.0,
  'conscience_score': 1.0},
 'runtime_meta': {'enabled': True, 'edits': 45, 'mode': 'live'},
 'baseline_preserved': True}

## 4) Production guardrail decision simulation

In [11]:
payload = {
    "action": "disclosed_pii_without_consent",
    "context": {"agent_did_this": True, "harm_occurred": True},
    "dry_run": True,
}

decision, status = decision_payload(payload)
{"status": status, "decision": decision["decision"]}


{'status': <HTTPStatus.OK: 200>,
 'decision': {'action': 'disclosed_pii_without_consent',
  'verdict': 'violation',
  'allowed': False,
  'norm_domain': 'privacy',
  'severity': 0.9,
  'penalty': 0.9,
  'risk_band': 'critical',
  'controls': ['block_action',
   'require_human_review',
   'log_decision',
   'redact_sensitive_content',
   'incident_alert']}}

## 5) Apply a binding update in service style (non-dry run)

In [12]:
payload_update = {
    "action": "deliberate_moderate_deception",
    "context": {"intent": "deliberate"},
    "dry_run": False,
    "apply_binding_update": True,
}

update_response, update_status = decision_payload(payload_update)
{
    "status": update_status,
    "mutated": update_response["state"]["mutated"],
    "norm_domain": update_response["decision"]["norm_domain"],
    "norm_before": update_response["state"]["norm_before"],
    "norm_after": update_response["state"]["norm_after"],
}


{'status': <HTTPStatus.OK: 200>,
 'mutated': False,
 'norm_domain': 'honesty',
 'norm_before': {'rule': 'Do not deceive, fabricate, or omit material information.',
  'weight': 1.0,
  'violations_count': 0,
  'threshold': 0.5},
 'norm_after': {'rule': 'Do not deceive, fabricate, or omit material information.',
  'weight': 1.0,
  'violations_count': 0,
  'threshold': 0.5}}